In [ ]:
import stata_setup
stata_setup.config("C:/Program Files/StataNow19/", "mp")

import sys
sys.path.append("C:/Program Files/StataNow19/utilities")
from pystata import config
config.init('mp')


In [ ]:
%%stata
log using "C:\Language_Interfacing\output\SurveyData.log", replace

clear
set obs 200
gen id=_n
set seed 12345
* Create survey design variables
gen psu = ceil(_n/10)                 // Primary sampling units
gen strata = mod(psu, 5) + 1          // 5 strata
gen weight = runiform(0.5, 2.5)        // Sampling weights
* Demographic variables
gen sex = (runiform() > 0.5)           // 0 = female, 1 = male
label define sexlbl 0 "Female" 1 "Male"
label values sex sexlbl
gen age = round(runiform(18, 65))
* Positive outcome variable (e.g., income, expenditure, exposure)
gen income = exp(rnormal(8, 0.6))      // log-normal distribution
label variable income "Monthly income"
label variable weight "Survey weight"
label variable psu "Primary sampling unit"
label variable strata "Stratum"
summ income
svyset [pw=weight]
gen ln_income = ln(income)
svy: mean ln_income,over(sex)
di exp( 7.9811)
di exp(8.030052)
*notice use of the ln variable (not the original)
svy: mean ln_income,over(sex)
ereturn display, eform (geo_mean)

*save dataset for export
keep id psu strata weight sex income
save "C:/Language_Interfacing/output/SurveyData.dta", replace

local date `c(current_date)'
local time `c(current_time)'
display _newline "Run `date' at `time'"

log close




In [ ]:
#do not use // in file paths.
import os
# Clear unsupported feature flags (e.g., sse3) that can break Polars/svy on some CPUs.
os.environ.pop("POLARS_FEATURE_FLAGS", None)
os.environ["POLARS_SKIP_CPU_CHECK"] = "1"
import svy
import pandas as pd
import numpy as np
from io import StringIO
import datetime as dt
import pyreadstat
import pyreadr
from pathlib import Path
path = Path("C:/Language_Interfacing/output")
file = "SurveyData.dta"
statafile = path / file
df = pd.read_stata(statafile)

In [ ]:
import polars as pl

#generate new column for log of income
df['ln_income'] = np.log(df['income'])

# Convert pandas DataFrame to Polars (required by svy)
df = pl.from_pandas(df)

#set the survey design variables
#Define the survey design
smp_design = svy.Design(
    stratum=("strata"),  
    psu="psu",                          # Primary sampling units
    wgt="weight",                          # Survey weights
)

#Create the sample object
sample = svy.Sample(data=df, design=smp_design)

In [ ]:
result = sample.estimation.mean("ln_income")
print(result)

In [ ]:
result = sample.estimation.mean(
    "ln_income",
    by="sex"
)
print(result)

In [ ]:
import datetime

print(f"Survey_data finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Pin rpy2 to R 4.6.1 before loading the extension.
import os
r_home = "C:/Program Files (x86)/R/R-4.6.1"
os.environ["R_HOME"] = r_home
os.environ["PATH"] = f"{r_home}/bin/x64;" + os.environ.get("PATH", "")
%load_ext rpy2.ipython
print("Configured R_HOME:", os.environ["R_HOME"])

In [ ]:
%%R
rm()
getwd()
.libPaths()
print(R.version.string)
if (!grepl("R version 4.6.1", R.version.string)) stop("R 4.6.1 is required. Restart kernel, then run the R setup cell first.")

drive<-"C:/Language_Interfacing"
folder<-"output"
setwd(paste0(drive, "/", folder))

#load libraries.
library(aod)
library(date)
library(lsr)
library(summarytools)
library(tibble)
library(haven)
library(dplyr)
library(ggplot2)
library(nnet)
library(epiR)
library(pwr)
library(psych)
library(logr)
library(stringr)
library(survey)

##Stata.
df<-haven::read_dta("C:/Language_Interfacing/output/SurveyData.dta")
dim(df)
is.data.frame(df)  # tibbles ARE data frames.
head(df)

#set the survey design.
design <- df %>%
  srvyr::as_survey(ids=psu,
                   weights = weight)
summary(design)

result <- design %>%
  dplyr::summarize(result = srvyr::survey_mean(income,
                                     na.rm = TRUE,
                                     vartype = c("se", "ci"))) %>%
as.data.frame(result)
print(result)

print(paste("Survey data finished at", format(Sys.time(),
                                              "%Y-%m-%d %H:%M:%S")))


In [ ]:
from git import Repo
from pathlib import Path

repo_root = Path(r"C:\Language_Interfacing")
repo = Repo(repo_root)

# Ensure target data files are ignored in git
gitignore_path = repo_root / ".gitignore"
ignore_entries = ["output/SurveyData.dta", "output/SurveyData.log"]

current_text = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""
existing_lines = {line.strip() for line in current_text.splitlines()}

for ignore_entry in ignore_entries:
    if ignore_entry not in existing_lines:
        separator = "" if (not current_text or current_text.endswith("\n")) else "\n"
        current_text = f"{current_text}{separator}{ignore_entry}\n"
        existing_lines.add(ignore_entry)
        print(f"Added to .gitignore: {ignore_entry}")
    else:
        print(f"Already in .gitignore: {ignore_entry}")

gitignore_path.write_text(current_text, encoding="utf-8")

In [ ]:
repo.index.add([r".gitignore", r"code\R_Stata_Python.ipynb"])
print("Staged: .gitignore and code\\R_Stata_Python.ipynb")

if repo.is_dirty(index=True, working_tree=True, untracked_files=False):
    commit_message = "Update R_Stata_Python.ipynb and gitignore"
    commit = repo.index.commit(commit_message)
    print(f"Committed: {commit.hexsha[:7]} - {commit_message}")
else:
    print("No tracked changes to commit")

origin = repo.remote(name="origin")
push_info = origin.push("main")[0]
print(f"Pushed to origin/main: {push_info.summary}")